In [6]:
from pathlib import Path
import pandas as pd

# =========================
# USER SETTINGS
# =========================
BASE_DIR = Path(r"C:/Plegma_Programming")
DATA_PATH = BASE_DIR / "PLEGMA_final_hourly_dataset.csv"
OUT_PATH = BASE_DIR / "Evaluation" / "House10" / "history_store.csv"

HOME_ID = "home10"
TARGET_DATE = "2026-05-30"   # ημερομηνία πρόβλεψης
HISTORY_HOURS =  168         # x ώρες
TARGET_YEAR = 2026

# Αν δεν υπάρχουν ακριβώς οι αντίστοιχες ημερομηνίες στο source year,
# τότε χρησιμοποιεί fallback: τις τελευταίες διαθέσιμες HISTORY_HOURS του σπιτιού
FALLBACK_TO_LATEST = True


# =========================
# LOAD DATA
# =========================
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

usecols = ["home_id", "timestamp", "consumption_Wh"]
df = pd.read_csv(DATA_PATH, usecols=usecols, parse_dates=["timestamp"], low_memory=False)

df = df.dropna(subset=["home_id", "timestamp", "consumption_Wh"]).copy()
df["home_id"] = df["home_id"].astype(str).str.strip()
df["consumption_Wh"] = pd.to_numeric(df["consumption_Wh"], errors="coerce")
df = df.dropna(subset=["consumption_Wh"]).copy()

df = df.sort_values(["home_id", "timestamp"]).reset_index(drop=True)

# κρατάμε μόνο το σπίτι
src = df[df["home_id"] == HOME_ID].copy()
if src.empty:
    raise ValueError(f"Δεν βρέθηκαν δεδομένα για το σπίτι: {HOME_ID}")

src = src.sort_values("timestamp").reset_index(drop=True)

# =========================
# TARGET WINDOW IN 2026
# =========================
target_end = pd.Timestamp(TARGET_DATE)          # π.χ. 2026-04-09 00:00
target_start = target_end - pd.Timedelta(hours=HISTORY_HOURS)

target_idx = pd.date_range(start=target_start, end=target_end, freq="h", inclusive="left")

# =========================
# BUILD SOURCE WINDOW
# =========================
def map_to_source_year(ts: pd.Timestamp, source_year: int) -> pd.Timestamp:
    return pd.Timestamp(
        year=source_year,
        month=ts.month,
        day=ts.day,
        hour=ts.hour
    )

source_years = sorted(src["timestamp"].dt.year.dropna().unique().tolist(), reverse=True)

best_source = None
best_year = None
best_coverage = -1

for year in source_years:
    mapped = [map_to_source_year(ts, year) for ts in target_idx]
    tmp = pd.DataFrame({"target_timestamp": target_idx, "source_timestamp": mapped})

    merged = tmp.merge(
        src[["timestamp", "consumption_Wh"]],
        left_on="source_timestamp",
        right_on="timestamp",
        how="left"
    )

    coverage = merged["consumption_Wh"].notna().sum()

    if coverage > best_coverage:
        best_coverage = coverage
        best_source = merged.copy()
        best_year = year

# =========================
# FALLBACK IF COVERAGE IS INCOMPLETE
# =========================
if best_source is None:
    raise ValueError("Αδυναμία δημιουργίας proxy history.")

if best_coverage < HISTORY_HOURS:
    if not FALLBACK_TO_LATEST:
        raise ValueError(
            f"Δεν βρέθηκαν αρκετές ωριαίες τιμές για το {HOME_ID}. "
            f"Coverage: {best_coverage}/{HISTORY_HOURS}"
        )

    if len(src) < HISTORY_HOURS:
        raise ValueError(
            f"Το σπίτι {HOME_ID} έχει μόνο {len(src)} ωριαίες εγγραφές, "
            f"λιγότερες από τις απαιτούμενες {HISTORY_HOURS}."
        )

    latest = src.tail(HISTORY_HOURS).copy().reset_index(drop=True)

    out = pd.DataFrame({
        "home_id": HOME_ID,
        "timestamp": target_idx,
        "consumption_Wh": latest["consumption_Wh"].astype(float).values
    })

    history_source = "fallback_latest_available"
else:
    out = pd.DataFrame({
        "home_id": HOME_ID,
        "timestamp": target_idx,
        "consumption_Wh": best_source["consumption_Wh"].astype(float).values
    })

    history_source = f"mapped_from_year_{best_year}"

# =========================
# FINAL SAVE
# =========================
out = out.sort_values("timestamp").reset_index(drop=True)
out.to_csv(OUT_PATH, index=False, encoding="utf-8")

print("Saved:", OUT_PATH)
print("Home:", HOME_ID)
print("Target date:", TARGET_DATE)
print("Rows:", len(out))
print("From:", out["timestamp"].min(), "To:", out["timestamp"].max())
print("History source:", history_source)

Saved: C:\Plegma_Programming\Evaluation\House10\history_store.csv
Home: home10
Target date: 2026-05-30
Rows: 168
From: 2026-05-23 00:00:00 To: 2026-05-29 23:00:00
History source: mapped_from_year_2023
